# Action-Angle Methods

galpy provides several methods for computing action-angle coordinates in
general (non-separable) potentials. Each method makes different approximations
and has different strengths. This tutorial covers:

1. **Staeckel approximation** (`actionAngleStaeckel`)
2. **Adiabatic approximation** (`actionAngleAdiabatic`)
3. **Orbit integration-based** (`actionAngleIsochroneApprox`)
4. **Reverse transformations** (`actionAngleTorus`)

In [1]:
%matplotlib inline
import numpy
import matplotlib.pyplot as plt
from galpy.potential import MWPotential2014

## 1. Staeckel Approximation

The Staeckel approximation (`actionAngleStaeckel`) is the most accurate general
method for computing actions in axisymmetric potentials. It works by locally
approximating the potential as a Staeckel potential (separable in confocal
ellipsoidal coordinates).

### Setup

The key parameter is `delta`, the focus of the confocal coordinate system.
You can estimate an appropriate value using `estimateDeltaStaeckel`:

In [2]:
from galpy.actionAngle import actionAngleStaeckel, estimateDeltaStaeckel

# Estimate delta for an orbit near R=1, z=0
delta = estimateDeltaStaeckel(MWPotential2014, 1.0, 0.0)
print("Estimated delta:", delta)

Estimated delta: 0.7013326670567281


In [3]:
aAS = actionAngleStaeckel(pot=MWPotential2014, delta=delta, c=True)

### Computing actions, frequencies, and angles

The interface is the same as for other action-angle classes:

In [4]:
# Just actions
jr, jphi, jz = aAS(1.0, 0.1, 1.1, 0.0, 0.05)
print(f"J_R = {jr.item():.6f}, L_z = {jphi.item():.6f}, J_z = {jz.item():.6f}")

J_R = 0.013602, L_z = 1.100000, J_z = 0.000464


In [5]:
# Full actions, frequencies, and angles
jr, jphi, jz, Or, Op, Oz, wr, wp, wz = aAS.actionsFreqsAngles(
    1.0, 0.1, 1.1, 0.0, 0.05, 0.0
)
print(f"J_R = {jr.item():.6f}")
print(
    f"Omega_R = {Or.item():.4f}, Omega_phi = {Op.item():.4f}, Omega_z = {Oz.item():.4f}"
)
print(
    f"theta_R = {wr.item():.4f}, theta_phi = {wp.item():.4f}, theta_z = {wz.item():.4f}"
)

J_R = 0.013602
Omega_R = 1.1595, Omega_phi = 0.8685, Omega_z = 2.2218
theta_R = 0.4696, theta_phi = 6.1767, theta_z = 6.0942


### Estimating delta along an orbit

The optimal `delta` depends on the orbit. For more accuracy, you can
estimate `delta` along an integrated orbit:

In [6]:
from galpy.orbit import Orbit

o = Orbit([1.0, 0.1, 1.1, 0.0, 0.05, 0.0])
ts = numpy.linspace(0, 50, 1001)
o.integrate(ts, MWPotential2014)

# Estimate delta from the orbit's R and z range
delta_orbit = estimateDeltaStaeckel(MWPotential2014, o.R(ts), o.z(ts))
print("Delta from orbit:", delta_orbit)

Delta from orbit: 0.7125563004146815


### Grid-based interpolation: actionAngleStaeckelGrid

For applications requiring many evaluations (e.g., distribution functions),
you can set up a grid-based interpolation that precomputes actions on a grid
and interpolates:

In [7]:
from galpy.actionAngle import actionAngleStaeckelGrid

aASG = actionAngleStaeckelGrid(
    pot=MWPotential2014, delta=delta, c=True, nE=51, npsi=51, nLz=61
)

# Same interface as actionAngleStaeckel
jr_grid, jphi_grid, jz_grid = aASG(1.0, 0.1, 1.1, 0.0, 0.05)
print(f"J_R (grid) = {jr_grid.item():.6f} vs J_R (direct) = {jr.item():.6f}")

J_R (grid) = 0.013602 vs J_R (direct) = 0.013602


## 2. Adiabatic Approximation

The adiabatic approximation (`actionAngleAdiabatic`) separates the radial
and vertical motions, computing $J_z$ at each radius while assuming the
radial motion is slow compared to the vertical oscillation. This is fast
but less accurate than the Staeckel method for orbits with large vertical
excursions.

In [8]:
from galpy.actionAngle import actionAngleAdiabatic

aAA = actionAngleAdiabatic(pot=MWPotential2014, c=True)

jr_ad, jphi_ad, jz_ad = aAA(1.0, 0.1, 1.1, 0.0, 0.05)
print(f"J_R = {jr_ad.item():.6f}, L_z = {jphi_ad.item():.6f}, J_z = {jz_ad.item():.6f}")

J_R = 0.013525, L_z = 1.100000, J_z = 0.000469


Compare to the Staeckel result:

In [9]:
print(f"J_R: Staeckel = {jr.item():.6f}, Adiabatic = {jr_ad.item():.6f}")
print(f"J_z: Staeckel = {jz.item():.6f}, Adiabatic = {jz_ad.item():.6f}")

J_R: Staeckel = 0.013602, Adiabatic = 0.013525
J_z: Staeckel = 0.000464, Adiabatic = 0.000469


### Grid-based: actionAngleAdiabaticGrid

Like the Staeckel method, the adiabatic approximation can be sped up
with a precomputed grid:

In [10]:
from galpy.actionAngle import actionAngleAdiabaticGrid

aAAG = actionAngleAdiabaticGrid(
    pot=MWPotential2014, c=True, nR=31, nEz=31, nEr=51, nLz=51
)

jr_ag, jphi_ag, jz_ag = aAAG(1.0, 0.1, 1.1, 0.0, 0.05)
print(f"J_R (grid) = {jr_ag.item():.6f}, J_z (grid) = {jz_ag.item():.6f}")

J_R (grid) = 0.013527, J_z (grid) = 0.000477


## 3. Orbit Integration-based: actionAngleIsochroneApprox

The `actionAngleIsochroneApprox` method (Bovy 2014) computes actions by
integrating the orbit and performing the action-angle calculation in a
best-fit isochrone potential. This is more general but slower than the
Staeckel method.

In [11]:
from galpy.actionAngle import actionAngleIsochroneApprox

aAIA = actionAngleIsochroneApprox(pot=MWPotential2014, b=0.8)

jr_ia, jphi_ia, jz_ia = aAIA(1.0, 0.1, 1.1, 0.0, 0.05, 0.0)
print(f"J_R = {jr_ia[0]:.6f}, L_z = {jphi_ia[0]:.6f}, J_z = {jz_ia[0]:.6f}")

J_R = 0.013629, L_z = 1.100000, J_z = 0.000466


In [12]:
# Compare all three methods
print("Method          J_R        J_z")
print(f"Staeckel:       {jr.item():.6f}   {jz.item():.6f}")
print(f"Adiabatic:      {jr_ad.item():.6f}   {jz_ad.item():.6f}")
print(f"IsochroneApprox:{jr_ia[0]:.6f}   {jz_ia[0]:.6f}")

Method          J_R        J_z
Staeckel:       0.013602   0.000464
Adiabatic:      0.013525   0.000469
IsochroneApprox:0.013629   0.000466


## 4. Reverse Transformations: actionAngleTorus

The `actionAngleTorus` class (using the Torus Mapper; McMillan & Binney 2008)
performs the **reverse** transformation: given actions and angles, it computes
phase-space coordinates $(\mathbf{x}, \mathbf{v})$.

This is useful for setting up initial conditions with specific actions, or for
computing orbital tori.

In [13]:
from galpy.actionAngle import actionAngleTorus

try:
    aAT = actionAngleTorus(pot=MWPotential2014)
    _has_torus = True
except RuntimeError:
    print("actionAngleTorus C extension not available; skipping torus examples")
    _has_torus = False

actionAngleTorus C extension not available; skipping torus examples


In [14]:
if _has_torus:
    # Specify actions and a set of angles
    jr_val = 0.01
    lz_val = 1.1
    jz_val = 0.001
    angler = numpy.linspace(0.0, 2.0 * numpy.pi, 101)
    anglephi = numpy.zeros(101)
    anglez = numpy.zeros(101)

    # Compute phase-space coordinates on this torus
    RvR = aAT(jr_val, lz_val, jz_val, angler, anglephi, anglez)

    # RvR contains [R, vR, vT, z, vz, phi]
    plt.plot(RvR[0], RvR[3])
    plt.xlabel(r"$R$")
    plt.ylabel(r"$z$")
    plt.title("Orbital torus cross-section (varying theta_R)")
else:
    print("Skipped: actionAngleTorus not available")

Skipped: actionAngleTorus not available


The `xvFreqs` method also returns the frequencies:

In [15]:
if _has_torus:
    result = aAT.xvFreqs(jr_val, lz_val, jz_val, angler, anglephi, anglez)
    # result = (R, vR, vT, z, vz, phi, OR, Op, Oz)
    print(f"Omega_R = {result[6]:.4f}")
    print(f"Omega_phi = {result[7]:.4f}")
    print(f"Omega_z = {result[8]:.4f}")
else:
    print("Skipped: actionAngleTorus not available")

Skipped: actionAngleTorus not available


## Method comparison summary

| Method | Speed | Accuracy | Frequencies/Angles | Notes |
|--------|-------|----------|-------------------|-------|
| `actionAngleStaeckel` | Fast (with C) | High for disk orbits | Yes | Best general method |
| `actionAngleAdiabatic` | Very fast | Good for thin-disk | Actions only | Best for quick estimates |
| `actionAngleIsochroneApprox` | Slow | High | Yes | Most general |
| `actionAngleTorus` | Moderate | High | Yes (reverse) | Reverse transformation only |
| `actionAngleIsochrone` | Very fast | Exact | Yes | Isochrone potential only |
| `actionAngleSpherical` | Fast | Exact | Yes | Spherical potentials only |